# Module 21 Lab: Real-time & Streaming Agents

Trong lab này, bạn sẽ xây dựng **Streaming Infrastructure cho LoanBot** — từ SSE simulator đến full backpressure-aware streaming system.

## Lab Objectives
1. Simulate SSE streaming với async generators
2. Implement streaming token consumer với metrics
3. Build WebSocket conversation manager với context window management
4. Handle streaming tool calls (accumulate + execute)
5. Implement backpressure với asyncio.Queue
6. Build streaming health monitor

In [ ]:
import asyncio, json, time, random
from dataclasses import dataclass, field
from typing import AsyncGenerator, List, Dict, Optional, Callable
from collections import defaultdict, deque
import re

print('✅ Setup complete')

## Section 1: SSE Stream Simulator

In [ ]:
@dataclass
class SSEEvent:
    event: str        # event type (message, start, done, tool_start)
    data: dict
    id: Optional[str] = None  # for Last-Event-ID resume

    def to_wire(self) -> str:
        """Format as SSE wire format: 'event: ...\ndata: ...\n\n'"""
        lines = []
        if self.id:
            lines.append(f'id: {self.id}')
        if self.event != 'message':
            lines.append(f'event: {self.event}')
        lines.append(f'data: {json.dumps(self.data)}')
        return '\n'.join(lines) + '\n\n'

async def simulate_loanbot_stream(
    app_id: str, credit_score: int, dti: float,
    token_delay: float = 0.05  # seconds between tokens
) -> AsyncGenerator[SSEEvent, None]:
    """Simulate LoanBot streaming analysis as SSE events."""
    event_id = 0

    def make_event(event_type: str, data: dict) -> SSEEvent:
        nonlocal event_id
        event_id += 1
        return SSEEvent(event=event_type, data=data, id=str(event_id))

    # Start event
    yield make_event('start', {'app_id': app_id, 'timestamp': time.time()})
    await asyncio.sleep(0.1)  # simulate initial processing

    # Stream analysis tokens
    analysis_parts = [
        f'Đang phân tích hồ sơ {app_id}... ',
        f'Credit score: {credit_score} ',
        f'({"✓ Đạt" if credit_score >= 620 else "✗ Dưới ngưỡng"} ngưỡng 620). ',
        f'DTI ratio: {dti:.0%} ',
        f'({"✓ Đạt" if dti <= 0.45 else "✗ Vượt"} ngưỡng 45%). ',
    ]

    for part in analysis_parts:
        for char in part:
            yield make_event('message', {'type': 'token', 'text': char})
            await asyncio.sleep(token_delay)

    # Tool call event
    yield make_event('tool_start', {'tool': 'query_cic', 'input': {'app_id': app_id}})
    await asyncio.sleep(0.3)  # simulate CIC query
    yield make_event('tool_done', {'tool': 'query_cic', 'result': {'history': 'clean', 'score': credit_score}})

    # Final decision
    if credit_score >= 620 and dti <= 0.45:
        decision = 'APPROVE'
    elif credit_score < 580 or dti > 0.60:
        decision = 'REJECT'
    else:
        decision = 'REVIEW'

    decision_text = f'\nQuyết định: {decision}'
    for char in decision_text:
        yield make_event('message', {'type': 'token', 'text': char})
        await asyncio.sleep(token_delay)

    yield make_event('done', {'decision': decision, 'app_id': app_id})

# Test: consume the stream and print events
async def test_sse_stream():
    print('=== SSE Stream Test (FC-2024-002) ===')
    full_text = ''
    start = time.time()
    ttfb = None

    async for event in simulate_loanbot_stream('FC-2024-002', credit_score=580, dti=0.58, token_delay=0.01):
        if event.event == 'start':
            print(f'  [start] {event.data}')
        elif event.event == 'message' and event.data.get('type') == 'token':
            if ttfb is None:
                ttfb = time.time() - start
            full_text += event.data['text']
        elif event.event == 'tool_start':
            print(f'  [tool_start] {event.data["tool"]}')
        elif event.event == 'tool_done':
            print(f'  [tool_done] result: {event.data["result"]}')
        elif event.event == 'done':
            print(f'  [done] decision={event.data["decision"]}')

    total = time.time() - start
    print(f'\nText: {full_text}')
    print(f'TTFB: {ttfb:.3f}s | Total: {total:.3f}s')
    print(f'UX improvement: TTFB is {total/ttfb:.1f}x faster than total')

await test_sse_stream()

## Section 2: Streaming Metrics Collector

In [ ]:
@dataclass
class StreamMetrics:
    session_id: str
    start_time: float = field(default_factory=time.time)
    ttfb: Optional[float] = None
    tokens_received: int = 0
    bytes_received: int = 0
    tool_calls: int = 0
    tool_latencies: List[float] = field(default_factory=list)
    end_time: Optional[float] = None
    dropped: bool = False

    @property
    def total_time(self) -> Optional[float]:
        return (self.end_time - self.start_time) if self.end_time else None

    @property
    def token_rate(self) -> float:
        elapsed = (self.end_time or time.time()) - self.start_time
        return self.tokens_received / elapsed if elapsed > 0 else 0

    @property
    def avg_tool_latency(self) -> Optional[float]:
        return sum(self.tool_latencies) / len(self.tool_latencies) if self.tool_latencies else None

    def report(self) -> dict:
        return {
            'session': self.session_id,
            'ttfb_ms': round(self.ttfb * 1000, 1) if self.ttfb else None,
            'total_ms': round(self.total_time * 1000, 1) if self.total_time else None,
            'tokens': self.tokens_received,
            'token_rate': round(self.token_rate, 1),
            'tool_calls': self.tool_calls,
            'avg_tool_latency_ms': round(self.avg_tool_latency * 1000, 1) if self.avg_tool_latency else None,
            'dropped': self.dropped
        }

async def stream_with_metrics(
    session_id: str, stream_gen
) -> StreamMetrics:
    """Consume a stream and collect metrics."""
    m = StreamMetrics(session_id=session_id)
    tool_start_time = None

    async for event in stream_gen:
        if event.event == 'message' and event.data.get('type') == 'token':
            if m.ttfb is None:
                m.ttfb = time.time() - m.start_time
            text = event.data.get('text', '')
            m.tokens_received += len(text.split()) or 1  # approx
            m.bytes_received += len(text.encode())
        elif event.event == 'tool_start':
            m.tool_calls += 1
            tool_start_time = time.time()
        elif event.event == 'tool_done':
            if tool_start_time:
                m.tool_latencies.append(time.time() - tool_start_time)
                tool_start_time = None
        elif event.event == 'done':
            m.end_time = time.time()

    return m

# Test
async def test_metrics():
    metrics = await stream_with_metrics(
        'sess-001',
        simulate_loanbot_stream('FC-2024-001', credit_score=720, dti=0.32, token_delay=0.005)
    )
    print('=== Stream Metrics ===')
    for k, v in metrics.report().items():
        print(f'  {k}: {v}')

await test_metrics()

## Section 3: WebSocket Conversation Manager

In [ ]:
@dataclass
class ConversationTurn:
    role: str   # 'user' or 'assistant'
    content: str
    timestamp: float = field(default_factory=time.time)

class ConversationManager:
    """Manage conversation history with context window limit."""

    def __init__(self, max_turns: int = 10, max_tokens_estimate: int = 4000):
        self.max_turns = max_turns
        self.max_tokens_estimate = max_tokens_estimate
        self.sessions: Dict[str, List[ConversationTurn]] = {}
        self.summaries: Dict[str, str] = {}  # compressed older context

    def add_turn(self, session_id: str, role: str, content: str):
        if session_id not in self.sessions:
            self.sessions[session_id] = []
        self.sessions[session_id].append(ConversationTurn(role=role, content=content))
        self._maybe_compress(session_id)

    def _maybe_compress(self, session_id: str):
        """Compress older turns into summary when history too long."""
        turns = self.sessions[session_id]
        if len(turns) > self.max_turns:
            # Compress oldest half
            compress_count = len(turns) - self.max_turns
            to_compress = turns[:compress_count]
            self.sessions[session_id] = turns[compress_count:]

            # Generate summary (in real system: call LLM to summarize)
            summary_parts = []
            for t in to_compress:
                prefix = 'Analyst' if t.role == 'user' else 'LoanBot'
                summary_parts.append(f'{prefix}: {t.content[:80]}...')
            new_summary = '; '.join(summary_parts)

            existing = self.summaries.get(session_id, '')
            self.summaries[session_id] = (existing + ' | ' + new_summary).strip(' | ')

    def get_messages(self, session_id: str) -> List[dict]:
        """Get messages list for Claude API, with summary prepended if exists."""
        messages = []
        turns = self.sessions.get(session_id, [])

        # Inject summary as first user message if exists
        summary = self.summaries.get(session_id)
        if summary:
            messages.append({
                'role': 'user',
                'content': f'[Context từ cuộc hội thoại trước: {summary}]'
            })
            messages.append({'role': 'assistant', 'content': 'Tôi đã ghi nhớ context trước đó.'})

        for turn in turns:
            messages.append({'role': turn.role, 'content': turn.content})

        return messages

    def estimate_tokens(self, session_id: str) -> int:
        """Rough token estimate: ~4 chars per token."""
        messages = self.get_messages(session_id)
        total_chars = sum(len(m['content']) for m in messages)
        return total_chars // 4

# Demo
manager = ConversationManager(max_turns=6)
session = 'analyst-hoang'

# Simulate a conversation about FC-2024-002
exchanges = [
    ('user', 'Phân tích hồ sơ FC-2024-002'),
    ('assistant', 'FC-2024-002: credit score 580 (dưới ngưỡng), DTI 58% (vượt ngưỡng). Quyết định: REVIEW.'),
    ('user', 'Tại sao REVIEW thay vì REJECT?'),
    ('assistant', 'Lịch sử tín dụng của KH khá tốt — chỉ DTI cao. Nếu KH có thu nhập phụ chứng minh được, có thể xét lại.'),
    ('user', 'Nếu score tăng 40 điểm thì quyết định thay đổi không?'),
    ('assistant', 'Score 620 vẫn borderline. Quan trọng hơn là DTI cần giảm về dưới 45%. Khuyến nghị: yêu cầu KH chứng minh thêm thu nhập.'),
    ('user', 'So sánh với FC-2024-003?'),
    ('assistant', 'FC-2024-003: score 400, DTI 75%, blacklisted → REJECT rõ ràng. FC-2024-002 tốt hơn nhiều, đáng xem xét human review.'),
]

print('=== Conversation Manager Demo ===')
for role, content in exchanges:
    manager.add_turn(session, role, content)

print(f'Total turns added: {len(exchanges)}')
print(f'Turns in memory: {len(manager.sessions[session])}')
print(f'Summary exists: {bool(manager.summaries.get(session))}')
if manager.summaries.get(session):
    print(f'Summary (first 150 chars): {manager.summaries[session][:150]}...')
print(f'Estimated tokens: {manager.estimate_tokens(session)}')
print(f'Messages for API: {len(manager.get_messages(session))} messages')

## Section 4: Streaming Tool Call Handler

In [ ]:
from enum import Enum

class StreamEventType(Enum):
    TEXT_DELTA    = 'text_delta'
    TOOL_START    = 'tool_start'
    TOOL_INPUT    = 'tool_input_delta'  # partial JSON
    TOOL_DONE     = 'tool_done'
    STREAM_DONE   = 'stream_done'

@dataclass
class MockStreamEvent:
    type: StreamEventType
    data: dict

async def mock_claude_with_tool_call(prompt: str) -> AsyncGenerator[MockStreamEvent, None]:
    """Simulate Claude streaming that includes a tool call."""
    # Phase 1: stream initial analysis text
    intro = 'Tôi sẽ truy vấn CIC để lấy lịch sử tín dụng.'
    for char in intro:
        yield MockStreamEvent(StreamEventType.TEXT_DELTA, {'text': char})
        await asyncio.sleep(0.005)

    # Phase 2: tool call starts
    yield MockStreamEvent(StreamEventType.TOOL_START, {
        'id': 'tool-001', 'name': 'query_cic'
    })

    # Phase 3: stream partial JSON for tool input
    tool_input_parts = ['{"app', '_id":', ' "FC-', '2024-', '002"}']
    for part in tool_input_parts:
        yield MockStreamEvent(StreamEventType.TOOL_INPUT, {'partial_json': part})
        await asyncio.sleep(0.01)

    # Phase 4: stop_reason = tool_use (caller handles this)
    yield MockStreamEvent(StreamEventType.STREAM_DONE, {'stop_reason': 'tool_use'})

class StreamingToolCallProcessor:
    """Process streaming events, handle tool calls, continue stream."""

    def __init__(self, tool_registry: dict):
        self.tools = tool_registry  # name → async function
        self.output_chunks: List[str] = []
        self.tool_calls_made: List[dict] = []

    async def process(self, stream_gen) -> str:
        text_buffer = ''
        tool_input_buffer = ''
        current_tool_id = None
        current_tool_name = None

        async for event in stream_gen:
            if event.type == StreamEventType.TEXT_DELTA:
                text = event.data['text']
                text_buffer += text
                self.output_chunks.append(text)  # would send via SSE

            elif event.type == StreamEventType.TOOL_START:
                current_tool_id = event.data['id']
                current_tool_name = event.data['name']
                tool_input_buffer = ''
                print(f'  🔧 Tool starting: {current_tool_name}')

            elif event.type == StreamEventType.TOOL_INPUT:
                tool_input_buffer += event.data['partial_json']

            elif event.type == StreamEventType.STREAM_DONE:
                if event.data.get('stop_reason') == 'tool_use' and current_tool_name:
                    # Execute tool
                    try:
                        tool_input = json.loads(tool_input_buffer)
                        print(f'  🔧 Executing {current_tool_name} with {tool_input}')
                        tool_fn = self.tools.get(current_tool_name)
                        if tool_fn:
                            result = await tool_fn(**tool_input)
                            self.tool_calls_made.append({
                                'id': current_tool_id, 'name': current_tool_name,
                                'input': tool_input, 'result': result
                            })
                            print(f'  ✅ Tool result: {result}')
                    except json.JSONDecodeError as e:
                        print(f'  ❌ Failed to parse tool input: {e} | buffer: {tool_input_buffer}')

        return text_buffer

# Define mock CIC tool
async def query_cic(app_id: str) -> dict:
    await asyncio.sleep(0.1)  # simulate API call
    return {'app_id': app_id, 'credit_history': 'clean', 'score': 580, 'overdue_count': 0}

# Test
async def test_tool_streaming():
    print('=== Streaming Tool Call Test ===')
    processor = StreamingToolCallProcessor(tool_registry={'query_cic': query_cic})
    text = await processor.process(mock_claude_with_tool_call('Analyze FC-2024-002'))
    print(f'  Text output: "{text}"')
    print(f'  Tool calls made: {len(processor.tool_calls_made)}')
    print(f'  Output chunks: {len(processor.output_chunks)}')

await test_tool_streaming()

## Section 5: Backpressure Implementation

In [ ]:
@dataclass
class BackpressureMetrics:
    tokens_produced: int = 0
    tokens_consumed: int = 0
    max_buffer_depth: int = 0
    backpressure_events: int = 0   # times producer had to wait
    slow_client_detected: bool = False
    start_time: float = field(default_factory=time.time)

    @property
    def producer_rate(self) -> float:
        elapsed = time.time() - self.start_time
        return self.tokens_produced / elapsed if elapsed > 0 else 0

    @property
    def consumer_rate(self) -> float:
        elapsed = time.time() - self.start_time
        return self.tokens_consumed / elapsed if elapsed > 0 else 0

async def buffered_stream_pipeline(
    producer_gen,
    consumer_fn: Callable,    # async fn(token: str) -> None
    max_buffer: int = 20,
    slow_threshold: float = 10.0,   # tokens/sec
    producer_timeout: float = 1.0   # max wait to put into buffer
) -> BackpressureMetrics:
    """Pipeline: producer → bounded queue → consumer. Apply backpressure automatically."""
    queue = asyncio.Queue(maxsize=max_buffer)
    metrics = BackpressureMetrics()

    async def producer():
        async for event in producer_gen:
            if event.type == StreamEventType.TEXT_DELTA:
                token = event.data['text']
                metrics.tokens_produced += 1
                try:
                    await asyncio.wait_for(queue.put(token), timeout=producer_timeout)
                    depth = queue.qsize()
                    if depth > metrics.max_buffer_depth:
                        metrics.max_buffer_depth = depth
                except asyncio.TimeoutError:
                    metrics.slow_client_detected = True
                    print(f'  ⚠️ Slow client detected — dropping stream')
                    break
        await queue.put(None)  # sentinel

    async def consumer():
        while True:
            token = await queue.get()
            if token is None:
                break
            await consumer_fn(token)
            metrics.tokens_consumed += 1

    await asyncio.gather(producer(), consumer())
    return metrics

# Test 1: Normal client (fast consumer)
async def fast_consumer(token: str):
    pass  # instant consumption

# Test 2: Slow client (slow consumer)
async def slow_consumer(token: str):
    await asyncio.sleep(0.05)  # 50ms per token = ~20 tokens/sec

async def test_backpressure():
    print('=== Backpressure Tests ===')

    # Fast consumer: no backpressure needed
    print('\n[Test 1: Fast consumer]')
    m1 = await buffered_stream_pipeline(
        mock_claude_with_tool_call('test'),
        fast_consumer, max_buffer=10
    )
    print(f'  Produced: {m1.tokens_produced}, Consumed: {m1.tokens_consumed}')
    print(f'  Max buffer depth: {m1.max_buffer_depth}')
    print(f'  Slow client detected: {m1.slow_client_detected}')

    # Slow consumer: backpressure kicks in
    print('\n[Test 2: Slow consumer (50ms/token)]')
    m2 = await buffered_stream_pipeline(
        mock_claude_with_tool_call('test'),
        slow_consumer, max_buffer=5, producer_timeout=0.3
    )
    print(f'  Produced: {m2.tokens_produced}, Consumed: {m2.tokens_consumed}')
    print(f'  Max buffer depth: {m2.max_buffer_depth}')
    print(f'  Slow client: {m2.slow_client_detected}')

await test_backpressure()

## Section 6: Streaming Health Monitor

In [ ]:
class StreamingHealthMonitor:
    """Monitor health of all active streaming sessions."""

    ALERT_THRESHOLDS = {
        'ttfb_p99_ms': 2000,         # alert if p99 TTFB > 2s
        'drop_rate_pct': 1.0,        # alert if >1% sessions dropped
        'buffer_depth_pct': 80.0,    # alert if buffer >80% full
        'max_active_sessions': 250,  # alert if approaching capacity
    }

    def __init__(self):
        self.completed_sessions: List[StreamMetrics] = []
        self.active_sessions: Dict[str, float] = {}  # session_id → start_time
        self.alerts: List[str] = []

    def session_start(self, session_id: str):
        self.active_sessions[session_id] = time.time()

    def session_end(self, metrics: StreamMetrics):
        self.active_sessions.pop(metrics.session_id, None)
        self.completed_sessions.append(metrics)
        self._check_alerts(metrics)

    def _check_alerts(self, latest: StreamMetrics):
        recent = self.completed_sessions[-50:]  # last 50 sessions

        if len(recent) >= 10:
            # TTFB p99
            ttfbs = sorted([s.ttfb * 1000 for s in recent if s.ttfb], reverse=True)
            p99_idx = max(0, int(len(ttfbs) * 0.01))
            p99_ttfb = ttfbs[p99_idx] if ttfbs else 0
            if p99_ttfb > self.ALERT_THRESHOLDS['ttfb_p99_ms']:
                self.alerts.append(f'⚠️ TTFB p99 {p99_ttfb:.0f}ms > {self.ALERT_THRESHOLDS["ttfb_p99_ms"]}ms')

            # Drop rate
            drop_count = sum(1 for s in recent if s.dropped)
            drop_rate = drop_count / len(recent) * 100
            if drop_rate > self.ALERT_THRESHOLDS['drop_rate_pct']:
                self.alerts.append(f'⚠️ Drop rate {drop_rate:.1f}% > {self.ALERT_THRESHOLDS["drop_rate_pct"]}%')

        # Active sessions
        if len(self.active_sessions) > self.ALERT_THRESHOLDS['max_active_sessions']:
            self.alerts.append(f'⚠️ Active sessions {len(self.active_sessions)} > capacity limit')

    def report(self) -> dict:
        recent = self.completed_sessions[-50:]
        if not recent:
            return {'status': 'no data'}

        ttfbs = [s.ttfb * 1000 for s in recent if s.ttfb]
        ttfbs.sort()

        def percentile(lst, p):
            if not lst: return 0
            idx = int(len(lst) * p / 100)
            return lst[min(idx, len(lst)-1)]

        return {
            'active_sessions': len(self.active_sessions),
            'completed_sessions': len(self.completed_sessions),
            'ttfb_p50_ms': round(percentile(ttfbs, 50), 1),
            'ttfb_p99_ms': round(percentile(ttfbs, 99), 1),
            'avg_token_rate': round(sum(s.token_rate for s in recent) / len(recent), 1),
            'drop_rate_pct': round(sum(1 for s in recent if s.dropped) / len(recent) * 100, 2),
            'avg_tool_calls': round(sum(s.tool_calls for s in recent) / len(recent), 1),
            'alerts': self.alerts[-5:]  # last 5 alerts
        }

# Demo
async def test_health_monitor():
    monitor = StreamingHealthMonitor()

    # Simulate 20 sessions
    for i in range(20):
        app_id = f'FC-2024-{i:03d}'
        score = random.randint(400, 800)
        dti = round(random.uniform(0.25, 0.65), 2)
        monitor.session_start(app_id)
        metrics = await stream_with_metrics(
            app_id,
            simulate_loanbot_stream(app_id, score, dti, token_delay=0.001)
        )
        # Randomly mark some as dropped
        if random.random() < 0.05:
            metrics.dropped = True
        monitor.session_end(metrics)

    print('=== Streaming Health Report ===')
    report = monitor.report()
    for k, v in report.items():
        print(f'  {k}: {v}')

await test_health_monitor()

## Section 7: Practice — SSE Resume với Last-Event-ID

**Bài tập:** Implement SSE resume: khi client reconnect với `Last-Event-ID: 5`, server phải replay events 6, 7, 8... từ server-side buffer.

Implement class `SSEEventBuffer` với:
- `add_event(event: SSEEvent)` — thêm event vào buffer
- `get_missed_events(last_event_id: str) -> List[SSEEvent]` — trả về events sau ID đó

In [ ]:
# TODO: Implement SSEEventBuffer
class SSEEventBuffer:
    def __init__(self, max_size: int = 1000):
        self.buffer = deque(maxlen=max_size)

    def add_event(self, event: SSEEvent):
        # TODO: add event to buffer
        pass

    def get_missed_events(self, last_event_id: Optional[str]) -> List[SSEEvent]:
        # TODO: return all events with id > last_event_id
        # if last_event_id is None, return all events
        return []

# SOLUTION:
# class SSEEventBuffer:
#     def __init__(self, max_size: int = 1000):
#         self.buffer = deque(maxlen=max_size)
#
#     def add_event(self, event: SSEEvent):
#         self.buffer.append(event)
#
#     def get_missed_events(self, last_event_id: Optional[str]) -> List[SSEEvent]:
#         if last_event_id is None:
#             return list(self.buffer)
#         try:
#             last_id = int(last_event_id)
#             return [e for e in self.buffer if e.id and int(e.id) > last_id]
#         except (ValueError, TypeError):
#             return list(self.buffer)

# Test (will pass when implemented)
buf = SSEEventBuffer()
for i in range(10):
    buf.add_event(SSEEvent('message', {'text': f'token-{i}'}, id=str(i+1)))

missed = buf.get_missed_events('5')
print(f'Events after ID 5: {len(missed)} (expected 5)')
if missed:
    print(f'First missed: ID={missed[0].id}, data={missed[0].data}')
else:
    print('💡 Implement SSEEventBuffer to see results!')